# Faza 1 — Model ML minim pentru estimarea prețului ureei

Obiectiv: construirea unui prim flux funcțional pentru predicția prețului ureei pe baza unei serii de timp lunare.

Modele incluse:
1. Naive Forecast — baseline;
2. LSTM;
3. BiLSTM.

Dataset folosit: World Bank Commodity Price Data / Pink Sheet, sheet-ul `Monthly Prices`, coloana `UREA_EE_BULK`.


In [ ]:
# 1. Importuri

import os
import re
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Reproducibilitate
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Pentru rulare mai stabilă pe laptop
torch.set_num_threads(1)


In [ ]:
# 2. Încărcarea datasetului

from pathlib import Path
import urllib.request

DATA_FILENAME = "CMO-Historical-Data-Monthly.xlsx"
WB_COMMODITY_PAGE = "https://www.worldbank.org/en/research/commodity-markets"
# URL curent Pink Sheet (se actualizează periodic pe site-ul World Bank)
WB_MONTHLY_URL = (
    "https://thedocs.worldbank.org/en/doc/74e8be41ceb20fa0da750cda2f6b9e4e-0050012026/"
    "related/CMO-Historical-Data-Monthly.xlsx"
)


def find_project_root() -> Path:
    """Găsește rădăcina proiectului (folderul care conține data/)."""
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd() / "notebooks",
        Path.cwd().parent / "notebooks",
    ]
    seen = set()
    for start in candidates:
        for root in (start, start.parent):
            root = root.resolve()
            if root in seen:
                continue
            seen.add(root)
            if (root / "data").is_dir():
                return root
    return Path.cwd().resolve()


def download_monthly_excel(dest: Path) -> None:
    """Descarcă Excel-ul Pink Sheet dacă lipsește local."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    headers = {"User-Agent": "Mozilla/5.0 (compatible; Disertatie-ML/1.0)"}

    def fetch(url: str) -> None:
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=120) as resp:
            dest.write_bytes(resp.read())

    try:
        fetch(WB_MONTHLY_URL)
        return
    except Exception:
        pass

    # Fallback: link din pagina Commodity Markets
    page_req = urllib.request.Request(WB_COMMODITY_PAGE, headers=headers)
    with urllib.request.urlopen(page_req, timeout=60) as resp:
        html = resp.read().decode("utf-8", errors="ignore")
    match = re.search(
        r"https://thedocs\.worldbank\.org/[^\"']+CMO-Historical-Data-Monthly\.xlsx",
        html,
    )
    if not match:
        raise FileNotFoundError(
            "Nu s-a putut descărca automat fișierul Excel.\n"
            f"Descarcă manual de la {WB_COMMODITY_PAGE}\n"
            f"și salvează-l în: {dest}"
        )
    fetch(match.group(0))


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / DATA_FILENAME

if not DATA_PATH.exists():
    print(f"Descărcare dataset în {DATA_PATH} ...")
    download_monthly_excel(DATA_PATH)
    print("Descărcare finalizată.")

prices = pd.read_excel(DATA_PATH, sheet_name="Monthly Prices", header=6)

prices = prices.rename(columns={"Unnamed: 0": "date_str"})
prices = prices.replace({"…": np.nan, "...": np.nan})

def parse_month(value):
    value = str(value)
    match = re.match(r"(\d{4})M(\d{2})", value)
    if match:
        return pd.Timestamp(year=int(match.group(1)), month=int(match.group(2)), day=1)
    return pd.NaT

prices["date"] = prices["date_str"].apply(parse_month)
prices["urea_price"] = pd.to_numeric(prices["UREA_EE_BULK"], errors="coerce")

df = (
    prices[["date", "urea_price"]]
    .dropna()
    .sort_values("date")
    .reset_index(drop=True)
)

df.head(), df.tail(), df.shape


In [ ]:
# 3. Verificare rapidă a seriei de timp

print("Prima lună:", df["date"].min())
print("Ultima lună:", df["date"].max())
print("Număr observații:", len(df))
print("Valori lipsă:", df["urea_price"].isna().sum())

plt.figure(figsize=(12, 5))
plt.plot(df["date"], df["urea_price"])
plt.title("Evoluția prețului ureei")
plt.xlabel("An")
plt.ylabel("Preț uree ($/mt)")
plt.grid(True)
plt.show()


In [ ]:
# 4. Împărțire cronologică train / validation / test

# Important: pentru serii de timp nu folosim random split.
series = df[["urea_price"]].values.astype("float32")
dates = df["date"].values

n = len(series)
train_end = int(n * 0.80)
val_end = int(n * 0.90)

train_raw = series[:train_end]
val_raw = series[train_end:val_end]
test_raw = series[val_end:]

print("Train:", df.iloc[0]["date"], "→", df.iloc[train_end-1]["date"], len(train_raw))
print("Validation:", df.iloc[train_end]["date"], "→", df.iloc[val_end-1]["date"], len(val_raw))
print("Test:", df.iloc[val_end]["date"], "→", df.iloc[-1]["date"], len(test_raw))


In [ ]:
# 5. Scalare

# Fit doar pe train pentru a evita data leakage.
scaler = MinMaxScaler()
scaler.fit(train_raw)

series_scaled = scaler.transform(series)


In [ ]:
# 6. Transformare în ferestre temporale

# Exemplu: ultimele 12 luni → luna următoare.
WINDOW_SIZE = 12

def create_windows(data, window_size):
    X, y, target_indices = [], [], []
    for i in range(window_size, len(data)):
        X.append(data[i-window_size:i])
        y.append(data[i])
        target_indices.append(i)
    return np.array(X, dtype="float32"), np.array(y, dtype="float32"), np.array(target_indices)

X, y, target_indices = create_windows(series_scaled, WINDOW_SIZE)

train_mask = target_indices < train_end
val_mask = (target_indices >= train_end) & (target_indices < val_end)
test_mask = target_indices >= val_end

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]
test_indices = target_indices[test_mask]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)


In [ ]:
# 7. Baseline: Naive Forecast

# Predicția pentru luna t = valoarea din luna t-1.
actual_test = series[test_indices].ravel()
naive_pred = series[test_indices - 1].ravel()

def regression_metrics(actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    rmse = math.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape}

baseline_metrics = regression_metrics(actual_test, naive_pred)
baseline_metrics


In [ ]:
# 8. Dataset PyTorch

class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(TimeSeriesDataset(X_train, y_train), batch_size=32, shuffle=False)


In [ ]:
# 9. Modele LSTM și BiLSTM

class LSTMRegressor(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1, bidirectional=False):
        super().__init__()
        self.bidirectional = bidirectional

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional
        )

        directions = 2 if bidirectional else 1

        self.fc = nn.Sequential(
            nn.Linear(hidden_size * directions, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        last_output = out[:, -1, :]
        return self.fc(last_output)


In [ ]:
# 10. Funcții de antrenare și evaluare

def train_model(model, train_loader, X_val, y_val, epochs=100, lr=0.001, patience=15):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_predictions = model(X_val_tensor)
            val_loss = criterion(val_predictions, y_val_tensor).item()

        avg_train_loss = np.mean(train_losses)
        history.append({"epoch": epoch + 1, "train_loss": avg_train_loss, "val_loss": val_loss})

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history)


def evaluate_model(model, X_test, y_test, scaler):
    model.eval()

    with torch.no_grad():
        predicted_scaled = model(torch.tensor(X_test, dtype=torch.float32)).numpy()

    predicted = scaler.inverse_transform(predicted_scaled).ravel()
    actual = scaler.inverse_transform(y_test).ravel()

    return regression_metrics(actual, predicted), predicted, actual


In [ ]:
# 11. Antrenare LSTM

torch.manual_seed(SEED)

lstm_model = LSTMRegressor(input_size=1, hidden_size=32, bidirectional=False)

lstm_model, lstm_history = train_model(
    lstm_model,
    train_loader,
    X_val,
    y_val,
    epochs=120,
    lr=0.001,
    patience=20
)

lstm_metrics, lstm_pred, actual = evaluate_model(lstm_model, X_test, y_test, scaler)

lstm_metrics


In [ ]:
# 12. Antrenare BiLSTM

torch.manual_seed(SEED)

bilstm_model = LSTMRegressor(input_size=1, hidden_size=32, bidirectional=True)

bilstm_model, bilstm_history = train_model(
    bilstm_model,
    train_loader,
    X_val,
    y_val,
    epochs=120,
    lr=0.001,
    patience=20
)

bilstm_metrics, bilstm_pred, actual = evaluate_model(bilstm_model, X_test, y_test, scaler)

bilstm_metrics


In [ ]:
# 13. Compararea modelelor

results = pd.DataFrame([
    {"Model": "Naive Forecast", **baseline_metrics},
    {"Model": "LSTM", **lstm_metrics},
    {"Model": "BiLSTM", **bilstm_metrics},
])

results


In [ ]:
# 14. Grafic: valori reale vs predicții

test_dates = df.loc[test_indices, "date"].values

plt.figure(figsize=(12, 5))
plt.plot(test_dates, actual, label="Valori reale")
plt.plot(test_dates, naive_pred, label="Naive Forecast")
plt.plot(test_dates, lstm_pred, label="LSTM")
plt.plot(test_dates, bilstm_pred, label="BiLSTM")
plt.title("Predicția prețului ureei: valori reale vs modele")
plt.xlabel("Dată")
plt.ylabel("Preț uree ($/mt)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# 15. Salvarea modelelor și a rezultatelor

torch.save(lstm_model.state_dict(), "lstm_urea_model.pt")
torch.save(bilstm_model.state_dict(), "bilstm_urea_model.pt")

results.to_csv("faza1_model_comparison.csv", index=False)
df.to_csv("faza1_urea_clean.csv", index=False)

print("Fișiere salvate:")
print("- lstm_urea_model.pt")
print("- bilstm_urea_model.pt")
print("- faza1_model_comparison.csv")
print("- faza1_urea_clean.csv")


## Interpretare inițială

După rulare, se analizează:

1. dacă LSTM și BiLSTM depășesc baseline-ul;
2. care model are MAE, RMSE și MAPE mai mici;
3. dacă fereastra de 12 luni este suficientă;
4. dacă este necesară extinderea modelului cu variabile externe: gaz natural, petrol, cereale, curs USD/EUR.

Această versiune este modelul minim. În Faza 2, modelul se poate extinde spre multivariate time series.
